# 1.2 风险与波动率

## 学习目标
- 理解金融风险的定义与度量方式
- 掌握波动率（Volatility）的计算
- 学会计算夏普比率（Sharpe Ratio）
- 理解最大回撤（Max Drawdown）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (12, 5)

# 下载数据
tickers = ['AAPL', 'TSLA', 'SPY']
data = yf.download(tickers, start='2021-01-01', end='2024-01-01',
                   progress=False)['Close']
returns = data.pct_change().dropna()
print('数据下载完成 ✅')
returns.tail(3)

## 1. 波动率（Volatility）

波动率 = 收益率的标准差，衡量资产价格的**不确定性**。

$$\sigma_{daily} = \text{std}(r_t)$$

$$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$$

In [ ]:
# 计算年化波动率
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)

print('年化波动率：')
for ticker in tickers:
    print(f'  {ticker:<6}: {annual_vol[ticker]:.2%}')

# 滚动波动率（20日）
rolling_vol = returns.rolling(20).std() * np.sqrt(252)

fig, ax = plt.subplots()
for ticker in tickers:
    ax.plot(rolling_vol.index, rolling_vol[ticker], label=ticker, linewidth=1.2)
ax.set_title('20日滚动年化波动率', fontsize=13)
ax.set_ylabel('波动率')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 夏普比率（Sharpe Ratio）

$$SR = \frac{\bar{r} - r_f}{\sigma}$$

衡量每单位风险所获得的**超额收益**，越高越好。
- $\bar{r}$：年化平均收益率
- $r_f$：无风险利率（常用美国国债利率，约 4-5%）
- $\sigma$：年化波动率

In [ ]:
risk_free_rate = 0.04  # 假设无风险利率 4%

annual_ret = returns.mean() * 252
sharpe = (annual_ret - risk_free_rate) / annual_vol

print(f'\n{"资产":<8} {"年化收益":<12} {"年化波动":<12} {"夏普比率"}')
print('-' * 45)
for ticker in tickers:
    print(f'{ticker:<8} {annual_ret[ticker]:<12.2%} {annual_vol[ticker]:<12.2%} {sharpe[ticker]:.2f}')

# 可视化
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if s > 0 else 'salmon' for s in sharpe.values]
bars = ax.bar(tickers, sharpe.values, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(1, color='green', linestyle='--', alpha=0.5, label='夏普=1（良好）')
ax.set_title('各资产夏普比率对比', fontsize=13)
ax.set_ylabel('Sharpe Ratio')
ax.legend()
for bar, val in zip(bars, sharpe.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 3. 最大回撤（Maximum Drawdown）

$$MDD = \max_{t} \left(\frac{\text{峰值} - \text{当前值}}{\text{峰值}}\right)$$

衡量从历史最高点到最低点的最大跌幅，是风险管理的重要指标。

In [ ]:
def max_drawdown(returns_series):
    """计算最大回撤"""
    cum_ret = (1 + returns_series).cumprod()
    rolling_max = cum_ret.cummax()
    drawdown = (cum_ret - rolling_max) / rolling_max
    return drawdown, drawdown.min()

fig, axes = plt.subplots(len(tickers), 1, figsize=(12, 10), sharex=True)

for ax, ticker in zip(axes, tickers):
    dd, mdd = max_drawdown(returns[ticker])
    ax.fill_between(dd.index, dd.values, 0, color='red', alpha=0.4)
    ax.plot(dd.index, dd.values, color='darkred', linewidth=0.8)
    ax.set_title(f'{ticker}  |  最大回撤: {mdd:.2%}', fontsize=11)
    ax.set_ylabel('回撤')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.grid(alpha=0.3)

plt.suptitle('各资产回撤曲线对比', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. 汇总风险指标

In [ ]:
summary = pd.DataFrame(index=tickers)
summary['年化收益率'] = annual_ret.map('{:.2%}'.format)
summary['年化波动率'] = annual_vol.map('{:.2%}'.format)
summary['夏普比率'] = sharpe.map('{:.2f}'.format)
summary['最大回撤'] = {t: f'{max_drawdown(returns[t])[1]:.2%}' for t in tickers}
summary

## 🎯 练习

1. 增加另一个资产（如债券 ETF `'TLT'`）并对比其风险特征。
2. 计算 2022 年熊市期间（2022-01-01 至 2022-12-31）的最大回撤。
3. 思考：**波动率高一定是坏事吗？** 结合夏普比率谈谈你的看法。

---
**下一节** → `03_market_basics.ipynb`